In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Literal

In [2]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("API_TOKEN"),
    base_url="https://openrouter.ai/api/v1"
)

In [3]:
@tool
def calculator(expression: str) -> str:
    """Calculate a math expression. Use for any arithmetic. Args: expression: e.g., '25 * 47'"""
    try:
        return f"{expression} = {eval(expression)}"
    except Exception as e:
        return f"Error: {str(e)}"

In [4]:
@tool
def search_web(query: str) -> str:
    """Search the web for information. Use for current events or facts. Args: query: search terms"""
    return f"Search results for '{query}': Several major developments reported this week."

In [5]:
tools = [calculator, search_web]
llm_with_tools = llm.bind_tools(tools)

In [6]:
tools_by_name = {t.name: t for t in tools}

In [7]:
tools_by_name

{'calculator': StructuredTool(name='calculator', description="Calculate a math expression. Use for any arithmetic. Args: expression: e.g., '25 * 47'", args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x000001E5D6896520>),
 'search_web': StructuredTool(name='search_web', description='Search the web for information. Use for current events or facts. Args: query: search terms', args_schema=<class 'langchain_core.utils.pydantic.search_web'>, func=<function search_web at 0x000001E5D6897B00>)}

In [8]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [9]:
def llm_node(state: State) -> dict:
    system = SystemMessage(content="You are a helpful assistant. Use tools when needed.")
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

In [10]:
def tool_node(state: State) -> dict:
    results = []
    for tc in state["messages"][-1].tool_calls:
        result = tools_by_name[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return {"messages": results}

In [11]:
def should_continue(state: State) -> Literal["tools", "__end__"]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return "__end__"


In [12]:
builder = StateGraph(State)
builder.add_node("llm", llm_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", should_continue, ["tools", "__end__"])
builder.add_edge("tools", "llm")

graph = builder.compile()

In [13]:
result = graph.invoke({
    "messages": [HumanMessage(content="What is 25 * 47 and what's the latest AI news?")]
})

print(result["messages"][-1].content)

The result of \( 25 \times 47 \) is \( 1175 \).

As for the latest AI news, there have been several major developments reported this week. If you would like more specific details or topics from the news, please let me know!
